## Notebook 5 – Reading & Writing Data

### Setup
For this notebook I need pandas and NumPy. Excel files need one extra library called openpyxl, and Parquet files need one called pyarrow. The command below installs all of them. If a library is already installed, pip just says "Requirement already satisfied", so it is safe to run again. I used %pip so that the libraries are installed in the same environment as this notebook.

The other things I use (os, glob, pathlib, sqlite3, pickle and json) come with Python, so they do not need to be installed.

In [1]:
%pip install pandas numpy openpyxl pyarrow

Defaulting to user installation because normal site-packages is not writeable
   ---------------------------------------- 0.0/28.6 MB ? eta -:--:--
   ---------------------------------------- 0.0/28.6 MB ? eta -:--:--
   ---------------------------------------- 0.0/28.6 MB ? eta -:--:--
   ---------------------------------------- 0.0/28.6 MB ? eta -:--:--
    --------------------------------------- 0.5/28.6 MB 2.4 MB/s eta 0:00:12
   - -------------------------------------- 0.8/28.6 MB 2.1 MB/s eta 0:00:14
   -- ------------------------------------- 2.1/28.6 MB 3.3 MB/s eta 0:00:09
   ----- ---------------------------------- 3.9/28.6 MB 4.7 MB/s eta 0:00:06
   -------- ------------------------------- 5.8/28.6 MB 5.7 MB/s eta 0:00:05
   ----------- ---------------------------- 7.9/28.6 MB 6.5 MB/s eta 0:00:04
   --------------- ------------------------ 10.7/28.6 MB 7.5 MB/s eta 0:00:03
   ------------------ --------------------- 13.4/28.6 MB 8.1 MB/s eta 0:00:02
   ---------------------


[notice] A new release of pip is available: 25.0.1 -> 26.2.1
[notice] To update, run: C:\Users\rafiy\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


Now I import everything that is needed for the whole notebook.

In [2]:
import os
import glob
import json
import time
import pickle
import sqlite3
from pathlib import Path

import numpy as np
import pandas as pd

I will use one small employee table in most of the topics. It has text, numbers and a date column, so I can check what happens to each data type when a file is saved and read again.

In [3]:
employees = pd.DataFrame({
    "Name": ["Asha", "Ravi", "Meena", "Kiran", "Sana", "Arjun"],
    "Age": [25, 32, 29, 41, 36, 28],
    "Department": ["HR", "IT", "IT", "Finance", "HR", "IT"],
    "Salary": [40000, 65000, 58000, 72000, 45000, 52000],
    "Joining_Date": pd.to_datetime(["2021-06-15", "2019-03-01", "2020-08-20",
                                    "2016-01-10", "2018-11-05", "2022-02-14"])
})

print(employees)
print()
print(employees.dtypes)

    Name  Age Department  Salary Joining_Date
0   Asha   25         HR   40000   2021-06-15
1   Ravi   32         IT   65000   2019-03-01
2  Meena   29         IT   58000   2020-08-20
3  Kiran   41    Finance   72000   2016-01-10
4   Sana   36         HR   45000   2018-11-05
5  Arjun   28         IT   52000   2022-02-14

Name                       str
Age                      int64
Department                 str
Salary                   int64
Joining_Date    datetime64[us]
dtype: object


## Topic 1 – CSV Files

### Step 1 – Understand the Concept
CSV stands for Comma Separated Values. It is a simple text file where every line is a row and the values in a line are separated by commas. The first line is usually the column names.

Pandas has read_csv() to read a CSV file into a DataFrame and to_csv() to save a DataFrame as a CSV file.

Some useful options of read_csv():
- sep : the separator, if it is not a comma (for example ";" or tab)
- usecols : read only the columns we want
- nrows : read only the first few rows
- index_col : use a column as the index
- parse_dates : read the given columns as dates
- header and names : used when the file has no header line

A CSV file keeps only text. It does not remember data types, so numbers and dates have to be recognised again when the file is read.

### Step 2 – Demonstrate the Concept

#### Business Example
A company exports its employee list from the HR software as a CSV file. The file can be opened in Excel by anyone, and it can also be read into Python for analysis.

### Step 3 – Implement the Concept

In [4]:
# Saving the table as CSV (index=False so that the row numbers are not saved)
employees.to_csv("employees.csv", index=False)

# Reading it back
df_csv = pd.read_csv("employees.csv")
print(df_csv)
print()
print(df_csv.dtypes)

    Name  Age Department  Salary Joining_Date
0   Asha   25         HR   40000   2021-06-15
1   Ravi   32         IT   65000   2019-03-01
2  Meena   29         IT   58000   2020-08-20
3  Kiran   41    Finance   72000   2016-01-10
4   Sana   36         HR   45000   2018-11-05
5  Arjun   28         IT   52000   2022-02-14

Name              str
Age             int64
Department        str
Salary          int64
Joining_Date      str
dtype: object


In [5]:
# Reading the date column as dates
df_dates = pd.read_csv("employees.csv", parse_dates=["Joining_Date"])
print(df_dates.dtypes)

# Reading only some columns and some rows
part = pd.read_csv("employees.csv", usecols=["Name", "Salary"], nrows=3)
print()
print(part)

Name                       str
Age                      int64
Department                 str
Salary                   int64
Joining_Date    datetime64[us]
dtype: object

    Name  Salary
0   Asha   40000
1   Ravi   65000
2  Meena   58000


In [6]:
# A file that uses a semicolon instead of a comma
employees.to_csv("employees_semicolon.csv", index=False, sep=";")
print(pd.read_csv("employees_semicolon.csv", sep=";"))

# A file that has no header line
employees.to_csv("employees_no_header.csv", index=False, header=False)
no_header = pd.read_csv("employees_no_header.csv", header=None, names=list(employees.columns))
print()
print(no_header)

    Name  Age Department  Salary Joining_Date
0   Asha   25         HR   40000   2021-06-15
1   Ravi   32         IT   65000   2019-03-01
2  Meena   29         IT   58000   2020-08-20
3  Kiran   41    Finance   72000   2016-01-10
4   Sana   36         HR   45000   2018-11-05
5  Arjun   28         IT   52000   2022-02-14

    Name  Age Department  Salary Joining_Date
0   Asha   25         HR   40000   2021-06-15
1   Ravi   32         IT   65000   2019-03-01
2  Meena   29         IT   58000   2020-08-20
3  Kiran   41    Finance   72000   2016-01-10
4   Sana   36         HR   45000   2018-11-05
5  Arjun   28         IT   52000   2022-02-14


#### What does this program do?
First I save the employees table to employees.csv with to_csv(). I used index=False so that the row numbers are not saved as an extra column. Then I read the file back with read_csv() and print the table and its data types.

In the output the table comes back with 6 rows and 5 columns. Name and Department are text and Age and Salary are int64, but Joining_Date comes back as text and not as a date, because CSV does not store data types. In the second cell parse_dates fixes this and Joining_Date becomes datetime64. The same cell also reads only the Name and Salary columns of the first 3 rows.

In the third cell I saved the file with a semicolon separator and read it again with sep=";". I also saved a file without a header line and read it with header=None and names=, so the column names were given by me.

#### AI/ML Example
Most datasets used for machine learning (for example the ones on Kaggle) are given as CSV files. The first step of a project is usually df = pd.read_csv("data.csv"). For a very big file, nrows and usecols can be used to load a small part first to look at the data.

#### Advantages
- Simple text file that almost every tool can open (Excel, Python, databases).
- Easy for a person to read and check.
- One line of code to read or write it.

#### Limitations
- It does not keep data types, so dates and numbers may need to be fixed after reading.
- Slow and big for very large data.
- A wrong separator gives a wrong table without any error.

#### Key Point
read_csv() reads a CSV file into a DataFrame and to_csv() saves it. Use index=False when saving and parse_dates when the file has dates.

## Topic 2 – Excel Files

### Step 1 – Understand the Concept
Excel files (.xlsx) can store many sheets in one file, along with formatting. Pandas can read them with read_excel() and save them with to_excel().

This needs the openpyxl library, which was installed in the setup.

Useful options:
- sheet_name : which sheet to read (name or number). sheet_name=None reads all the sheets into a dictionary of DataFrames.
- usecols, nrows, header : similar to read_csv()
- pd.ExcelWriter : used to write more than one sheet into the same file
- pd.ExcelFile : used to look at the sheet names of a file

### Step 2 – Demonstrate the Concept

#### Business Example
A finance team sends a monthly report as one Excel file. One sheet has the full employee list and another sheet has a short salary summary. Pandas can read any one sheet or all the sheets.

### Step 3 – Implement the Concept

In [7]:
# Saving one table to an Excel file
employees.to_excel("employees.xlsx", index=False, sheet_name="Employees")

df_excel = pd.read_excel("employees.xlsx")
print(df_excel)
print()
print(df_excel.dtypes)

    Name  Age Department  Salary Joining_Date
0   Asha   25         HR   40000   2021-06-15
1   Ravi   32         IT   65000   2019-03-01
2  Meena   29         IT   58000   2020-08-20
3  Kiran   41    Finance   72000   2016-01-10
4   Sana   36         HR   45000   2018-11-05
5  Arjun   28         IT   52000   2022-02-14

Name                       str
Age                      int64
Department                 str
Salary                   int64
Joining_Date    datetime64[us]
dtype: object


In [8]:
# Saving two sheets in one file
salary_summary = employees[["Name", "Salary"]]

with pd.ExcelWriter("hr_report.xlsx") as writer:
    employees.to_excel(writer, sheet_name="Employees", index=False)
    salary_summary.to_excel(writer, sheet_name="Salary_Summary", index=False)

# Looking at the sheet names
excel_file = pd.ExcelFile("hr_report.xlsx")
print("Sheets:", excel_file.sheet_names)

# Reading one sheet
print()
print(pd.read_excel("hr_report.xlsx", sheet_name="Salary_Summary"))

# Reading all sheets at once (gives a dictionary)
all_sheets = pd.read_excel("hr_report.xlsx", sheet_name=None)
print()
print(type(all_sheets))
print(list(all_sheets.keys()))

Sheets: ['Employees', 'Salary_Summary']

    Name  Salary
0   Asha   40000
1   Ravi   65000
2  Meena   58000
3  Kiran   72000
4   Sana   45000
5  Arjun   52000

<class 'dict'>
['Employees', 'Salary_Summary']


#### What does this program do?
The first cell saves the employees table into employees.xlsx with to_excel() and reads it back with read_excel(). Unlike CSV, the Joining_Date column should come back as datetime64, because Excel stores dates as dates.

In the second cell I made a second table with only the Name and Salary columns. Then pd.ExcelWriter is used to put both tables in one file called hr_report.xlsx, each on its own sheet. After that, ExcelFile shows the sheet names (Employees and Salary_Summary), read_excel with sheet_name reads one sheet, and sheet_name=None reads all the sheets into a dictionary where the keys are the sheet names and the values are DataFrames.

#### AI/ML Example
Data collected by other teams (for example survey results or sales numbers) very often comes as Excel files. The data scientist reads the needed sheet with read_excel(), cleans it and then uses it for analysis or model training.

#### Advantages
- Many sheets in one file.
- Data types like dates are kept.
- Easy to share with people who do not use Python.

#### Limitations
- Slower to read and write than CSV or Parquet.
- Needs an extra library (openpyxl).
- Not good for very large data (an Excel sheet has a limit of about 1 million rows).

#### Key Point
read_excel() and to_excel() work with Excel files. Use sheet_name to choose a sheet and pd.ExcelWriter to save many sheets in one file.

## Topic 3 – JSON Files

### Step 1 – Understand the Concept
JSON stands for JavaScript Object Notation. It is a text format that stores data as key-value pairs and lists, and it is the main format used by web APIs. JSON can also be nested, which means a value can itself contain another set of key-value pairs.

Pandas has read_json() and to_json(). The orient option decides the layout of the JSON:
- records : a list of rows, like [{"Name": "Asha", "Age": 25}, ...] (the most common one)
- columns : a dictionary of columns (this is the default for a DataFrame)

For nested JSON, pd.json_normalize() flattens it into a normal table.

### Step 2 – Demonstrate the Concept

#### Business Example
An online shop gets its orders from a web service in JSON format. Each order has the customer details inside it, which makes it nested. The shop converts it to a table to analyse it.

### Step 3 – Implement the Concept

In [9]:
# Saving as JSON (a list of records). Dates are saved as text in ISO format.
employees.to_json("employees.json", orient="records", date_format="iso", indent=4)

# Looking at the file
print(open("employees.json").read())

[
    {
        "Name":"Asha",
        "Age":25,
        "Department":"HR",
        "Salary":40000,
        "Joining_Date":"2021-06-15T00:00:00.000"
    },
    {
        "Name":"Ravi",
        "Age":32,
        "Department":"IT",
        "Salary":65000,
        "Joining_Date":"2019-03-01T00:00:00.000"
    },
    {
        "Name":"Meena",
        "Age":29,
        "Department":"IT",
        "Salary":58000,
        "Joining_Date":"2020-08-20T00:00:00.000"
    },
    {
        "Name":"Kiran",
        "Age":41,
        "Department":"Finance",
        "Salary":72000,
        "Joining_Date":"2016-01-10T00:00:00.000"
    },
    {
        "Name":"Sana",
        "Age":36,
        "Department":"HR",
        "Salary":45000,
        "Joining_Date":"2018-11-05T00:00:00.000"
    },
    {
        "Name":"Arjun",
        "Age":28,
        "Department":"IT",
        "Salary":52000,
        "Joining_Date":"2022-02-14T00:00:00.000"
    }
]


In [10]:
# Reading it back
df_json = pd.read_json("employees.json")
print(df_json)
print()
print(df_json.dtypes)

# Asking pandas to read the date column as dates
df_json_dates = pd.read_json("employees.json", convert_dates=["Joining_Date"])
print()
print(df_json_dates.dtypes)

    Name  Age Department  Salary             Joining_Date
0   Asha   25         HR   40000  2021-06-15T00:00:00.000
1   Ravi   32         IT   65000  2019-03-01T00:00:00.000
2  Meena   29         IT   58000  2020-08-20T00:00:00.000
3  Kiran   41    Finance   72000  2016-01-10T00:00:00.000
4   Sana   36         HR   45000  2018-11-05T00:00:00.000
5  Arjun   28         IT   52000  2022-02-14T00:00:00.000

Name              str
Age             int64
Department        str
Salary          int64
Joining_Date      str
dtype: object

Name                       str
Age                      int64
Department                 str
Salary                   int64
Joining_Date    datetime64[us]
dtype: object


In [11]:
# The default layout (orient="columns") compared with records
small = employees[["Name", "Age"]].head(2)
print(small.to_json())
print(small.to_json(orient="records"))

{"Name":{"0":"Asha","1":"Ravi"},"Age":{"0":25,"1":32}}
[{"Name":"Asha","Age":25},{"Name":"Ravi","Age":32}]


In [12]:
# Nested JSON
orders = [
    {"order_id": 1, "item": "Pen", "customer": {"name": "Asha", "city": "Hyderabad"}},
    {"order_id": 2, "item": "Book", "customer": {"name": "Ravi", "city": "Chennai"}},
    {"order_id": 3, "item": "Bag", "customer": {"name": "Meena", "city": "Pune"}}
]

print(pd.DataFrame(orders))
print()
print(pd.json_normalize(orders))

   order_id  item                               customer
0         1   Pen  {'name': 'Asha', 'city': 'Hyderabad'}
1         2  Book    {'name': 'Ravi', 'city': 'Chennai'}
2         3   Bag      {'name': 'Meena', 'city': 'Pune'}

   order_id  item customer.name customer.city
0         1   Pen          Asha     Hyderabad
1         2  Book          Ravi       Chennai
2         3   Bag         Meena          Pune


#### What does this program do?
The first cell saves the table as a JSON file with to_json(). I used orient="records" so that every row becomes one dictionary, date_format="iso" so that dates are saved as readable text like 2021-06-15T00:00:00.000, and indent=4 so that the file is easy to read. Then the file content is printed.

The second cell reads the file with read_json(). By default the Joining_Date column usually comes back as text, so I used convert_dates=["Joining_Date"] to get it as datetime64.

The third cell shows the two layouts. The default layout is a dictionary of columns where the inner keys are the row numbers, and the records layout is a list of rows.

In the last cell, the orders have a customer dictionary inside each order. If the list is put directly into a DataFrame, the customer column holds the whole dictionary. pd.json_normalize() opens it and makes separate columns named customer.name and customer.city.

#### AI/ML Example
Data collected from web APIs (like weather, social media or an app's backend) comes as JSON, often nested. json_normalize() turns it into a flat table that can be used for analysis or for building a dataset. JSON is also used for saving settings and for text datasets such as chatbot conversations.

#### Advantages
- The standard format of web APIs.
- Can store nested data.
- Readable by humans and by almost every programming language.

#### Limitations
- Bigger and slower than binary formats like Parquet.
- Nested data needs flattening before it can be used as a table.
- Dates and data types are not stored properly, so they need to be fixed after reading.

#### Key Point
Use read_json() and to_json() for JSON files, orient="records" for a list of rows, and json_normalize() to flatten nested JSON.

## Topic 4 – Parquet Files

### Step 1 – Understand the Concept
Parquet is a binary file format made for big data. It stores data column by column, compresses it, and keeps the data types. Because of this, Parquet files are usually smaller and faster to read than CSV files, and we can read only the columns we need.

Pandas has read_parquet() and to_parquet(). This needs the pyarrow library, which was installed in the setup.

### Step 2 – Demonstrate the Concept

#### Business Example
A company records millions of website visits every day. Saving these logs as CSV would take a lot of space and time. Saving them as Parquet makes the files smaller and the reports faster.

### Step 3 – Implement the Concept

In [13]:
# Saving and reading the employees table
employees.to_parquet("employees.parquet", index=False)

df_parquet = pd.read_parquet("employees.parquet")
print(df_parquet)
print()
print(df_parquet.dtypes)

# Reading only some columns
print()
print(pd.read_parquet("employees.parquet", columns=["Name", "Salary"]))

    Name  Age Department  Salary Joining_Date
0   Asha   25         HR   40000   2021-06-15
1   Ravi   32         IT   65000   2019-03-01
2  Meena   29         IT   58000   2020-08-20
3  Kiran   41    Finance   72000   2016-01-10
4   Sana   36         HR   45000   2018-11-05
5  Arjun   28         IT   52000   2022-02-14

Name                       str
Age                      int64
Department                 str
Salary                   int64
Joining_Date    datetime64[us]
dtype: object

    Name  Salary
0   Asha   40000
1   Ravi   65000
2  Meena   58000
3  Kiran   72000
4   Sana   45000
5  Arjun   52000


In [14]:
# Comparing CSV and Parquet on a bigger table (100,000 rows)
np.random.seed(42)
n = 100000
big = pd.DataFrame({
    "id": np.arange(n),
    "value": np.random.rand(n),
    "category": np.random.choice(["A", "B", "C"], n)
})

big.to_csv("big.csv", index=False)
big.to_parquet("big.parquet", index=False)

print("CSV size     :", os.path.getsize("big.csv") / 1024, "KB")
print("Parquet size :", os.path.getsize("big.parquet") / 1024, "KB")

start = time.perf_counter()
pd.read_csv("big.csv")
csv_time = time.perf_counter() - start

start = time.perf_counter()
pd.read_parquet("big.parquet")
parquet_time = time.perf_counter() - start

print()
print("CSV read time     :", round(csv_time, 4), "seconds")
print("Parquet read time :", round(parquet_time, 4), "seconds")

CSV size     : 2749.837890625 KB
Parquet size : 1595.6904296875 KB

CSV read time     : 0.1158 seconds
Parquet read time : 0.032 seconds


#### What does this program do?
The first cell saves the employees table as a Parquet file and reads it back. The output shows that Joining_Date is still datetime64 and all the other types are the same as before, because Parquet keeps data types. The cell also reads only the Name and Salary columns using the columns option.

In the second cell I created a bigger table with 100,000 rows using NumPy random numbers (with a seed so it can be repeated). It is saved as both CSV and Parquet, and then the file sizes and the reading times are printed using os.path.getsize() and time.perf_counter().

The Parquet file should be smaller and usually faster to read than the CSV file. The exact numbers depend on the computer, so they will be different on each system.

#### AI/ML Example
Large ML datasets and feature tables are commonly stored as Parquet, because they load faster and take less space. Reading only the needed columns saves memory too. Many big data tools like Spark and cloud data platforms also use Parquet.

#### Advantages
- Smaller files because of compression.
- Fast to read, and it is possible to read only some columns.
- Keeps data types.

#### Limitations
- It is a binary file, so it cannot be opened and read in a text editor or Excel.
- Needs an extra library (pyarrow).
- Not worth it for very small files.

#### Key Point
Parquet is a fast, compressed format that keeps data types. Use read_parquet() and to_parquet() for big datasets.

## Topic 5 – Pickle Files

### Step 1 – Understand the Concept
Pickle is a Python format that saves any Python object (a DataFrame, a list, a dictionary, a NumPy array, even a trained model) exactly as it is, and loads it back later. This process is called serialization.

Pandas has to_pickle() and read_pickle() for DataFrames. For other Python objects, the pickle module is used with pickle.dump() to save and pickle.load() to load. The file is opened in binary mode ("wb" to write and "rb" to read).

A pickle file should be opened only if it comes from a trusted source, because loading a pickle file can run code that is hidden in it.

### Step 2 – Demonstrate the Concept

#### Business Example
A report needs a long cleaning process that takes 20 minutes. After cleaning, the table is saved as a pickle file, so the next time the report can load the cleaned table in a second without doing the whole cleaning again.

### Step 3 – Implement the Concept

In [15]:
# Saving and loading a DataFrame
employees.to_pickle("employees.pkl")

df_pickle = pd.read_pickle("employees.pkl")
print(df_pickle)
print()
print(df_pickle.dtypes)
print()
print("Same as the original table?", df_pickle.equals(employees))

    Name  Age Department  Salary Joining_Date
0   Asha   25         HR   40000   2021-06-15
1   Ravi   32         IT   65000   2019-03-01
2  Meena   29         IT   58000   2020-08-20
3  Kiran   41    Finance   72000   2016-01-10
4   Sana   36         HR   45000   2018-11-05
5  Arjun   28         IT   52000   2022-02-14

Name                       str
Age                      int64
Department                 str
Salary                   int64
Joining_Date    datetime64[us]
dtype: object

Same as the original table? True


In [16]:
# Saving other Python objects with the pickle module
settings = {"model": "linear_regression", "learning_rate": 0.01, "features": ["Age", "Salary"]}

with open("settings.pkl", "wb") as file:
    pickle.dump(settings, file)

with open("settings.pkl", "rb") as file:
    loaded_settings = pickle.load(file)

print(loaded_settings)
print("Same as the original?", loaded_settings == settings)

{'model': 'linear_regression', 'learning_rate': 0.01, 'features': ['Age', 'Salary']}
Same as the original? True


#### What does this program do?
The first cell saves the employees table with to_pickle() and loads it again with read_pickle(). The output shows the same table, the same data types (Joining_Date is still datetime64) and True for equals(), which means the loaded table is exactly the same as the original.

The second cell saves a normal Python dictionary with pickle.dump() and loads it with pickle.load(). The files are opened in binary mode. The output shows the same dictionary and True for the comparison.

#### AI/ML Example
After a long data preparation step, the processed DataFrame is saved with pickle, so it is not repeated every time. Pickle is also the common way to save a trained machine learning model (for example a Scikit-learn model) and load it later for predictions.

#### Advantages
- Saves any Python object exactly as it is, with all data types.
- Very easy to use.
- Fast for saving and loading.

#### Limitations
- Only Python can read it, so it is not good for sharing with other tools.
- It is not safe to load pickle files from unknown sources.
- A pickle file saved with one version of Python or pandas may not load in a very different version.

#### Key Point
Pickle saves Python objects exactly as they are. Use to_pickle() and read_pickle() for DataFrames, and open pickle files only from trusted sources.

## Topic 6 – SQL Tables

### Step 1 – Understand the Concept
Most companies store their data in databases, and the data is taken out using SQL (Structured Query Language) queries. Pandas can read the result of a SQL query straight into a DataFrame using read_sql(), and can save a DataFrame as a table using to_sql().

In this notebook I use SQLite, because it comes with Python (the sqlite3 module) and the whole database is just one file, so nothing else needs to be installed.

Useful options:
- to_sql(name, connection, if_exists=..., index=False) : if_exists can be "fail" (default), "replace" or "append"
- read_sql(query, connection, params=..., parse_dates=...) : params passes values safely into the query

For other databases like MySQL or PostgreSQL, the SQLAlchemy library and a database driver are also needed (for example %pip install sqlalchemy pymysql). The idea of read_sql() stays the same.

### Step 2 – Demonstrate the Concept

#### Business Example
A company keeps its employee table in a database. A manager wants only the people from the IT department who earn more than 50000. Instead of loading everything, the query asks the database for only those rows, and pandas puts the result in a DataFrame.

### Step 3 – Implement the Concept

In [17]:
# Creating a connection (this makes the file company.db if it does not exist)
conn = sqlite3.connect("company.db")

# Saving the DataFrame as a table
rows_saved = employees.to_sql("employees", conn, if_exists="replace", index=False)
print("Rows saved:", rows_saved)

# Reading the full table
df_sql = pd.read_sql("SELECT * FROM employees", conn)
print()
print(df_sql)
print()
print(df_sql.dtypes)

Rows saved: 6

    Name  Age Department  Salary         Joining_Date
0   Asha   25         HR   40000  2021-06-15 00:00:00
1   Ravi   32         IT   65000  2019-03-01 00:00:00
2  Meena   29         IT   58000  2020-08-20 00:00:00
3  Kiran   41    Finance   72000  2016-01-10 00:00:00
4   Sana   36         HR   45000  2018-11-05 00:00:00
5  Arjun   28         IT   52000  2022-02-14 00:00:00

Name              str
Age             int64
Department        str
Salary          int64
Joining_Date      str
dtype: object


In [18]:
# Reading with a condition and sorting
query = "SELECT Name, Salary FROM employees WHERE Salary > 50000 ORDER BY Salary DESC"
print(pd.read_sql(query, conn))

# Passing a value safely with params
it_team = pd.read_sql("SELECT * FROM employees WHERE Department = ?", conn, params=("IT",))
print()
print(it_team)

# Reading the date column as dates
df_sql_dates = pd.read_sql("SELECT * FROM employees", conn, parse_dates=["Joining_Date"])
print()
print(df_sql_dates.dtypes)

# Closing the connection when the work is finished
conn.close()

    Name  Salary
0  Kiran   72000
1   Ravi   65000
2  Meena   58000
3  Arjun   52000

    Name  Age Department  Salary         Joining_Date
0   Ravi   32         IT   65000  2019-03-01 00:00:00
1  Meena   29         IT   58000  2020-08-20 00:00:00
2  Arjun   28         IT   52000  2022-02-14 00:00:00

Name                       str
Age                      int64
Department                 str
Salary                   int64
Joining_Date    datetime64[us]
dtype: object


#### What does this program do?
The first cell connects to a SQLite database file called company.db, saves the employees table in it using to_sql() (if_exists="replace" means an old table with the same name is replaced), and reads it back with a SELECT query.

The output shows the 6 rows. Joining_Date comes back as text, because SQLite stores dates as text.

The second cell reads only the Name and Salary of the people with a salary above 50000, sorted from the highest salary. The output should show Kiran, Ravi, Meena and Arjun in that order. The next query uses ? and params to read only the IT department, which gives Ravi, Meena and Arjun. Using params is safer than joining the value into the query text. Then parse_dates is used to get Joining_Date as datetime64. At the end the connection is closed with conn.close().

#### AI/ML Example
In companies, the training data for a model usually sits in a database. A SQL query selects and joins the needed data, and read_sql() loads it into a DataFrame for analysis and training. The final predictions can also be written back to the database with to_sql().

#### Advantages
- Reads exactly the rows and columns needed, so less data is loaded in memory.
- Works with the data where it is stored, without exporting files.
- The same pandas code works for many databases.

#### Limitations
- Needs a database and a connection (and extra libraries for databases other than SQLite).
- Data types may change (for example dates come back as text).
- The person needs to know basic SQL.

#### Key Point
read_sql() runs a SQL query and gives a DataFrame, and to_sql() saves a DataFrame as a table. Close the connection after use.

## Topic 7 – Reading Multiple Files

### Step 1 – Understand the Concept
Often the data is split into many files, for example one file for every month. To read all of them, the steps are:

1. Find the file names with glob.glob() using a pattern like "sales_*.csv" (the star means "anything").
2. Read each file with a loop or a list comprehension.
3. Join all the tables into one with pd.concat(). Using ignore_index=True gives a fresh row numbering.

It is also useful to add a column that tells which file each row came from.

### Step 2 – Demonstrate the Concept

#### Business Example
A shop has a separate sales file for each month: sales_jan.csv, sales_feb.csv and sales_mar.csv. For the quarter report, all three files are combined into one table.

### Step 3 – Implement the Concept
First I create three small monthly files, so there is something to read.

In [19]:
monthly_units = {
    "jan": [120, 135, 150],
    "feb": [110, 140, 160],
    "mar": [130, 125, 170]
}

for month, units in monthly_units.items():
    month_df = pd.DataFrame({"Product": ["Pen", "Book", "Bag"], "Units": units})
    month_df.to_csv("sales_" + month + ".csv", index=False)

print("Files created.")

Files created.


In [20]:
# Step 1: find the files
files = sorted(glob.glob("sales_*.csv"))
print(files)

# Step 2: read every file and remember its month
frames = []
for file in files:
    data = pd.read_csv(file)
    data["Month"] = file.replace("sales_", "").replace(".csv", "")
    frames.append(data)

# Step 3: join them
all_sales = pd.concat(frames, ignore_index=True)
print()
print(all_sales)
print()
print("Shape:", all_sales.shape)
print("Total units sold:", all_sales["Units"].sum())

['sales_feb.csv', 'sales_jan.csv', 'sales_mar.csv']

  Product  Units Month
0     Pen    110   feb
1    Book    140   feb
2     Bag    160   feb
3     Pen    120   jan
4    Book    135   jan
5     Bag    150   jan
6     Pen    130   mar
7    Book    125   mar
8     Bag    170   mar

Shape: (9, 3)
Total units sold: 1240


In [21]:
# The same thing in one line with a list comprehension
combined = pd.concat([pd.read_csv(f) for f in files], ignore_index=True)
print(combined.shape)

(9, 2)


#### What does this program do?
The first cell creates three small CSV files, one for each month, using a loop.

In the second cell, glob.glob("sales_*.csv") finds all the matching files and sorted() puts them in order. The output list should be sales_feb.csv, sales_jan.csv and sales_mar.csv, because sorted() arranges them alphabetically and not by month. Then a loop reads each file and adds a Month column from the file name. At last pd.concat() joins the three tables into one table with 9 rows and 3 columns, and the total units sold is printed.

The third cell does the same reading in a single line with a list comprehension. It gives the same 9 rows, but without the Month column because I did not add it there.

#### AI/ML Example
Big datasets are often stored as many part files (like part1.csv, part2.csv ...) or as one file per day. The same pattern of glob, read and concat is used to load all of them into one DataFrame before training a model.

#### Advantages
- Reads any number of files with a few lines of code.
- The Month (source) column keeps track of where each row came from.

#### Limitations
- All the files need to be loaded in memory together, which can be a problem for very large data.
- The files should have the same columns, otherwise the combined table will have missing values.
- sorted() sorts file names as text, so the order may not be the month order.

#### Key Point
Use glob.glob() to find the files, read each one, and join them with pd.concat(ignore_index=True).

## Topic 8 – File Path Handling

### Step 1 – Understand the Concept
A file path tells Python where a file is.

- Absolute path : the full address from the root of the drive, like C:/Users/name/project/data.csv. It works only on that computer.
- Relative path : the address from the folder where the notebook is running, like data/employees.csv. It works on any computer if the folder structure is the same.

Useful tools:
- os.getcwd() : the current working folder
- os.listdir() : the names of files and folders in a folder
- os.path.exists(path) : checks whether a path exists
- os.path.join("data", "file.csv") : joins path parts with the correct separator for the operating system
- os.makedirs(path, exist_ok=True) : creates a folder (and does not give an error if it exists)
- pathlib.Path : a newer, object style way to work with paths. The / operator joins parts, and it has name, suffix, parent, exists() and glob().

On Windows, a backslash path should be written as a raw string like r"C:\Users\name\data.csv", or with forward slashes, because a single backslash is an escape character in Python.

### Step 2 – Demonstrate the Concept

#### Real-World Example
An absolute path is like a full home address with the country, city and street, and a relative path is like saying "the second house on this street". The second one only makes sense if you already know where you are standing.

### Step 3 – Implement the Concept

In [22]:
# Where am I, and what is in this folder?
print("Current folder:", os.getcwd())

csv_files = [f for f in os.listdir() if f.endswith(".csv")]
print()
print("CSV files here:", csv_files)

# Checking if a file exists
print()
print("employees.csv exists?  ", os.path.exists("employees.csv"))
print("missing_file.csv exists?", os.path.exists("missing_file.csv"))

Current folder: C:\Users\rafiy\Documents\Sprint 3 – NumPy & Pandas for AIML Engineers

CSV files here: ['big.csv', 'employees.csv', 'employees_no_header.csv', 'employees_saved.csv', 'employees_semicolon.csv', 'employees_with_index.csv', 'sales_feb.csv', 'sales_jan.csv', 'sales_mar.csv']

employees.csv exists?   True
missing_file.csv exists? False


In [23]:
# Building paths with os.path.join and creating folders
raw_path = os.path.join("data", "raw")
processed_path = os.path.join("data", "processed")
os.makedirs(raw_path, exist_ok=True)
os.makedirs(processed_path, exist_ok=True)

file_path = os.path.join(raw_path, "employees.csv")
employees.to_csv(file_path, index=False)
print("Saved to:", file_path)
print(pd.read_csv(file_path))

Saved to: data\raw\employees.csv
    Name  Age Department  Salary Joining_Date
0   Asha   25         HR   40000   2021-06-15
1   Ravi   32         IT   65000   2019-03-01
2  Meena   29         IT   58000   2020-08-20
3  Kiran   41    Finance   72000   2016-01-10
4   Sana   36         HR   45000   2018-11-05
5  Arjun   28         IT   52000   2022-02-14


In [24]:
# The same with pathlib
output_file = Path("data") / "processed" / "employees_clean.csv"
employees.to_csv(output_file, index=False)

print("Path       :", output_file)
print("File name  :", output_file.name)
print("Extension  :", output_file.suffix)
print("Parent     :", output_file.parent)
print("Exists?    :", output_file.exists())
print("Full path  :", output_file.resolve())

# All CSV files in a folder
print()
for f in sorted(Path("data").glob("**/*.csv")):
    print(f)

Path       : data\processed\employees_clean.csv
File name  : employees_clean.csv
Extension  : .csv
Parent     : data\processed
Exists?    : True
Full path  : C:\Users\rafiy\Documents\Sprint 3 – NumPy & Pandas for AIML Engineers\data\processed\employees_clean.csv

data\processed\employees_clean.csv
data\raw\employees.csv


In [25]:
# What happens when the path is wrong
try:
    pd.read_csv("data/wrong_folder/employees.csv")
except FileNotFoundError as error:
    print("FileNotFoundError:", error)

FileNotFoundError: [Errno 2] No such file or directory: 'data/wrong_folder/employees.csv'


#### What does this program do?
The first cell prints the current working folder with os.getcwd(), lists the CSV files in it with os.listdir() and a list comprehension, and checks if a file exists with os.path.exists(). It prints True for employees.csv (created in Topic 1) and False for missing_file.csv.

In the second cell, os.path.join() builds the paths data/raw and data/processed (with the correct separator for the system), os.makedirs() creates the folders, and a CSV file is saved inside data/raw and read back.

The third cell does the same job with pathlib. The / operator joins the path parts, and then the file name, extension, parent folder, existence and full absolute path are printed. The loop at the end prints all the CSV files inside the data folder and its sub folders.

In the last cell I read from a folder that does not exist, and the FileNotFoundError is caught, so the message is printed and the notebook does not stop.

#### AI/ML Example
ML projects are organised in folders like data/raw, data/processed and models. Using relative paths with os.path.join() or pathlib makes the code work on any computer, so a teammate or a reviewer can run the same notebook without changing the paths.

#### Advantages
- Relative paths make the code portable across computers.
- os.path.join() and pathlib take care of the separators for Windows, Linux and Mac.
- Checking if a path exists avoids errors in the middle of the program.

#### Limitations
- Relative paths depend on the folder where the notebook is running, so they break if the notebook is run from another folder.
- Absolute paths do not work on another computer.

#### Key Point
Use relative paths with os.path.join() or pathlib.Path instead of typing full paths by hand, and check os.getcwd() when a file is not found.

## Interview Questions

#### 1. What is the difference between a CSV file and a Parquet file?

CSV is a plain text file that does not keep data types and is slow and big for large data. Parquet is a binary, compressed, column based format that keeps data types, is smaller and faster to read, and lets us read only some columns.

#### 2. Why do we use index=False in to_csv()?

By default to_csv() also writes the row numbers as the first column. With index=False they are not saved, so when the file is read again there is no extra "Unnamed: 0" column.

#### 3. How can you read a date column as dates from a CSV file?

Use the parse_dates option, for example pd.read_csv("file.csv", parse_dates=["Joining_Date"]). Without it, the dates are read as text because CSV does not store data types.

#### 4. How do you read all the sheets of an Excel file?

Use pd.read_excel("file.xlsx", sheet_name=None). It returns a dictionary where the keys are the sheet names and the values are DataFrames. Excel files need the openpyxl library.

#### 5. What is the difference between the orient="records" and the default orient in to_json()?

With orient="records" the JSON is a list of rows, where each row is a dictionary. The default for a DataFrame is "columns", which is a dictionary of columns where the inner keys are the row numbers.

#### 6. How do you convert nested JSON into a table?

Use pd.json_normalize(). It flattens the inner dictionaries into separate columns, with names like customer.name.

#### 7. What is pickle and what is the risk of using it?

Pickle saves any Python object exactly as it is and loads it back later, so it is used to save DataFrames and trained models. The risk is that loading a pickle file can run hidden code, so it should only be loaded from trusted sources.

#### 8. How do you read data from a SQL database into a DataFrame?

Make a connection (for example sqlite3.connect("db.sqlite")) and use pd.read_sql("SELECT ...", conn). A DataFrame can be saved as a table with df.to_sql("table", conn). For databases like MySQL or PostgreSQL, SQLAlchemy is used for the connection.

#### 9. How do you read many files of the same format into one DataFrame?

Find the files with glob.glob("pattern*.csv"), read each one (in a loop or a list comprehension), and join them with pd.concat(list_of_dataframes, ignore_index=True).

#### 10. What is the difference between an absolute path and a relative path? Why is os.path.join() better than typing the path as a string?

An absolute path is the full address from the root of the drive and works only on that computer. A relative path starts from the current working folder and is portable. os.path.join() puts the correct separator for the operating system, so the same code works on Windows, Linux and Mac.